# DCGAN

## Imports

In [ ]:
#%matplotlib inline
import os
import re
import gc
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import json
import time
import copy

from IPython.display import display, Image as Img
from PIL import Image
import cv2
from torch.utils.data import TensorDataset


# Set random seed for reproducibility
manualSeed = 999
#manualSeed = random.randint(1, 10000) # use if you want new results
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.cuda.manual_seed(manualSeed)
torch.use_deterministic_algorithms(True) # Needed for reproducible results

print(torch.cuda.is_available())         # True
print(torch.version.cuda)                # '11.8'
print(torch.cuda.get_device_name(0))     # 'NVIDIA GeForce RTX 4070 SUPER'

print(torch.backends.cudnn.version())    # Debería mostrar algo como 8700
print(torch.backends.cudnn.enabled)      # True



## Parameters

In [ ]:
# Root directory for dataset
# use of os.path.join to maximize campatibility beetween different Operating Systems
IMAGENETTE = "imagenette2" 
IMAGEWOOF = "imagewoof2"
GENERATED = os.path.join("data", "generated")
datarootImagenette = os.path.join("data", "databases", IMAGENETTE)
datarootImagewoof =  os.path.join("data", "databases", IMAGEWOOF)
PLOTS = "plots"
MODELS = "models"

# Directory for transformed datasets, rescaled to 64x64 or 128x128, in order to make the training faster (no cpu time wasted in resizing images)
TRANSFORMED_DATASET = "transformed_datasets"

# Number of epochs to save generated images
ROUNDS_PER_SAVE_IMAGES = 50

# Number of epochs to save models
ROUNDS_PER_SAVE_MODELS = 50

# Number of images to save for each epoch
NUM_IMAGES_FOR_STEP = 32 


# Number of workers for dataloader
workers = 16

# Batch size during training
batch_size = 128

# Number of channels in the training images. For color images this is 3
nc = 3

# Size of z latent vector (i.e. size of generator input)
nz = 100

# Size of feature maps in generator
ngf = 64

# Size of feature maps in discriminator
ndf = 64

# Beta1 hyperparameter for Adam optimizers
beta1 = 0.5

# Number of GPUs available. Use 0 for CPU mode.
ngpu = 1

# Learning rates values
LEARNING_RATES = [
    0.00009,
    0.0002,
    0.0005,
]

# To be developed
NZ_VALUES = [
    100,
    200,
]

NUM_EPOCHS = 500


In [ ]:
import os
import re

# Ignorar por extensión
ignored_exts = {'.jpeg', '.pdf', '.pt', '.txt', 'png', '.tgz'}

# Ignorar por nombre exacto
ignored_names = {'.gitkeep'}

# Regex para ignorar archivos tipo n######## (como WordNet IDs)
wnid_pattern = re.compile(r'^n\d{8}$')

def print_tree(start_path, prefix=""):
    items = sorted(os.listdir(start_path))
    for item in items:
        path = os.path.join(start_path, item)

        # Ignorar por nombre exacto
        if item in ignored_names:
            continue

        # Ignorar por patrón tipo "n########"
        name_no_ext = os.path.splitext(item)[0]
        if wnid_pattern.fullmatch(name_no_ext):
            continue

        # Ignorar por extensión
        if os.path.isfile(path):
            ext = os.path.splitext(item)[1].lower()
            if ext in ignored_exts:
                continue
            print(f"{prefix}└── {item}")
        elif os.path.isdir(path):
            print(f"{prefix}├── {item}")
            print_tree(path, prefix + "│   ")

# Ruta raíz
root = "data"
print(f"/{root}")
print_tree(root)


Imagenette is a subset of 10 easily classified classes from Imagenet (tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute).

Imagewoof is a subset of 10 classes from Imagenet that aren't so easy to classify, since they're all dog breeds. The breeds are: Australian terrier, Border terrier, Samoyed, Beagle, Shih-Tzu, English foxhound, Rhodesian ridgeback, Dingo, Golden retriever, Old English sheepdog.

In [ ]:
classificatorImagenette = {
    "tench": "n01440764", # Type of fish
    "English springer": "n02102040", # Type of dog
    "cassette player": "n02979186", 
    "chain saw": "n03000684",
    "church": "n03028079",
    "French horn": "n03394916",
    "garbage truck": "n03417042",
    "gas pump": "n03425413",
    "golf ball": "n03445777",
    "parachute": "n03888257"
}
classificatorImagenette_inv = {valor: clave for clave, valor in classificatorImagenette.items()}

classificatorImagewoof = {
    "Australian terrier": "n02086240", 
    "Border terrier": "n02087394", 
    "Samoyed": "n02088364", 
    "Beagle": "n02089973",
    "Shih-Tzu": "n02093754",
    "English foxhound": "n02096294",
    "Rhodesian ridgeback": "n02099601",
    "Dingo": "n02105641",
    "Golden retriever": "n02111889",
    "Old English sheepdog": "n02115641"
}

classificatorImagewoof_inv = {valor: clave for clave, valor in classificatorImagewoof.items()}



## Show examples of Images
Functions to make the dataset visuabel and make a better understanding of the data we are gonna use

In [ ]:
def show_images_dataset(dataroot, folderClassificator):
    images = []
    titles = []
    # Itera a través de cada categoría en el clasificador
    for key in folderClassificator:
        first_image_path = None
        folder_path = os.path.join(dataroot, "train", folderClassificator[key])

        titles.append(f"{key}, Samples: {len([_ for _ in os.listdir(folder_path)])}")

        # Encuentra la primera imagen en la carpeta
        for file_name in os.listdir(folder_path):
            if file_name.endswith(('.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG')):
                first_image_path = os.path.join(folder_path, file_name)
                break  # Detenemos el bucle en la primera imagen encontrada

        # Si se encontró la imagen, la añadimos a la lista
        if first_image_path:
            image = Image.open(first_image_path)
            images.append(image)
            
        else:
            print(f"No se encontró ninguna imagen en la carpeta para {key}")


    # Configuración del layout para mostrar las imágenes en una cuadrícula de 3 columnas
    num_images = len(images)
    num_rows = (num_images + 2) // 3  # Redondea hacia arriba para completar filas

    fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows), constrained_layout=True)  # Ocupa todo el ancho con ajuste automático

    # Dibuja cada imagen en la cuadrícula
    for i, ax in enumerate(axes.flat):
        if i < num_images:
            ax.imshow(images[i])
            ax.set_title(titles[i], fontsize=8)
            ax.axis('off')  # Quita los ejes
        else:
            ax.axis('off')  # Oculta celdas vacías

    plt.show(fig)

def most_common(lst):
    return max(set(lst), key=lst.count)

def get_image_dimension(path):
    im = cv2.imread(path)
    h, w, _ = im.shape
    return h, w

# Under construction
def create_heapmap_from_dimension(widths, heights):
    total = len(widths)
    dimensions = []
    maximum = max(max(widths), max(heights)) +1
    for _ in range(maximum):
        dimensions.append([])
    for i in range(maximum):
        for _ in range(maximum):
            dimensions[i].append(0)

    
    for i in range(total):
        dimensions[widths[i]][heights[i]] += 1

    # dimensions = gaussian_filter(dimensions, sigma=10)

    def heatmap2d(arr: np.ndarray):
        plt.imshow(arr, cmap='viridis')
        plt.colorbar()
        plt.show()

    heatmap2d(dimensions)

def show_dimensions_dataset(dataroot, folderClassificator, saveFile=None):
    imagesSize = []
    # Itera a través de cada categoría en el clasificador
    for key in folderClassificator:
        image_path = None
        folder_path = os.path.join(dataroot, "train", folderClassificator[key])
        # Save the dimension of every photo
        for file_name in os.listdir(folder_path):
            if file_name.endswith(('.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG')):
                image_path = os.path.join(folder_path, file_name)
                imagesSize.append(get_image_dimension(image_path))


    widths, heights = zip(*imagesSize)
    
    # Min, max, most common and average widths
    dataWidths = [min(widths), max(widths), most_common(widths), round(sum(widths)/len(widths))]
    print(f"Width:\n -Lowest:  {dataWidths[0]}, Highest:  {dataWidths[1]}, Most common: {dataWidths[2]}, Average: {dataWidths[3]}")

    # Min, max, most common and average heights
    dataHeights = [min(heights), max(heights), most_common(heights), round(sum(heights)/len(heights))]
    print(f"Height:\n -Lowest:  {dataHeights[0]}, Highest:  {dataHeights[1]}, Most common: {dataHeights[2]}, Average: {dataHeights[3]}")

    # Guardar números y datos adicionales como JSON
    if saveFile is not None:
        with open(saveFile+".txt", "w") as file:
            json.dump({"widths": dataWidths, "heights": dataHeights}, file)

    # Create the scatter plot
    fig = plt.figure(figsize=(10, 6))
    plt.xscale('log')  # Logarithmic scale for x-axis
    plt.yscale('log')  # Logarithmic scale for y-axis
    plt.scatter(widths, heights, marker='o', s=100)  # s controls the size of points
    plt.title("Image Dimensions (log scale)")
    plt.xlabel("Width (pixels) (log scale)")
    plt.ylabel("Height (pixels) (log scale)")
    
    # Custom formatter to avoid scientific notation on both axes
    ax = plt.gca()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda val, pos: f'{val:.0f}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, pos: f'{val:.0f}'))

    # Annotate each point with its width and height
    for i, (w, h) in enumerate(imagesSize):
        plt.annotate(f"({w}, {h})", (w, h), textcoords="offset points", xytext=(5, 5), ha='center')

    plt.grid(True)
    if saveFile is not None:
        plt.savefig(saveFile+".png")
        print("Saved dimensions")
    plt.show()
    

def load_dimensions_dataset(dataroot, folderClassificator, output_path):
    """
    Load or compute dimensions dataset.

    Parameters:
        output_path (str): Path to save/load the output file.
        dataroot (str): Root directory of the dataset.
        folderClassificator (dict): Dictionary mapping class names to folder names.

    Returns:
        None
    """
    if not os.path.exists(output_path+".png"):
        # Compute and save dimensions dataset
        fig = show_dimensions_dataset(dataroot, folderClassificator, saveFile=output_path)
    else:
        # Load the precomputed dimensions dataset
        with open(output_path+".txt", "r") as file:
            loaded_data = json.load(file)

        dataWidths = loaded_data["widths"]
        print(f"Width:\n -Lowest:  {dataWidths[0]}, Highest:  {dataWidths[1]}, Most common: {dataWidths[2]}, Average: {dataWidths[3]}")

        dataHeights = loaded_data["heights"]
        print(f"Height:\n -Lowest:  {dataHeights[0]}, Highest:  {dataHeights[1]}, Most common: {dataHeights[2]}, Average: {dataHeights[3]}")

        display(Img(filename=output_path+".png"))

    


### imaginette 
#### Image examples
Show images of every type of imaginette

In [ ]:
show_images_dataset(datarootImagenette,classificatorImagenette)

#### Distribution
Distribution of the images dimensions to see how the size looks like (A good idea to consider to wich size should be resized)

In [ ]:
output_path = os.path.join(GENERATED, PLOTS, "imagenette_dimension_distribution")

load_dimensions_dataset(datarootImagenette,classificatorImagenette, output_path)


### imagewoof 
#### Image examples

Show all the images of imagewoof

In [ ]:
show_images_dataset(datarootImagewoof,classificatorImagewoof)

#### Distribution

In [ ]:
output_path = os.path.join(GENERATED, PLOTS, "imagewoof_dimension_distribution")

load_dimensions_dataset(datarootImagenette,classificatorImagenette, output_path)

Function to merge datasets

In [ ]:
# Function to merge datasets

def merge_datasets(d1, d2):
    return TensorDataset(d1[:][0],d2[:])

## Create a dataset of the images.


In [ ]:
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

def show_images_grid(dataset, dataloader, titlePlot):# Plot some training images
        
    # Número total de imágenes en el dataset
    total_images = len(dataset)
    print(f"Total de imágenes en el dataset: {total_images}")

    # Número de batches por epoch en el dataloader
    batches_per_epoch = len(dataloader)
    print(f"Número de batches por epoch: {batches_per_epoch}")


    real_batch = next(iter(dataloader))
    plt.figure(figsize=(8,8))
    plt.axis("off")
    plt.title(titlePlot)
    plt.imshow(np.transpose(vutils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),(1,2,0)))
    plt.show()


We use a class to create the dataset transforming the images when we create the dataset instead of when we iterate with them. Doing this we only transform the images once as we will train with the same dataset multiple models. The other way around the images where getting transformed every time we train a different model with the same dataset. 

In [ ]:
class PreloadedImageFolderClasses(dset.ImageFolder):
    def __init__(self, root, transform=None, target_transform=None):
        super().__init__(root, transform=None, target_transform=target_transform)
        
        # Aplicamos las transformaciones en __init__
        self.transform = transform
        self.preloaded_data = []
        
        for path, target in self.samples:
            sample = self.loader(path)  # Cargar la imagen
            if self.transform:
                sample = self.transform(sample)  # Aplicar la transformación
            self.preloaded_data.append((sample, target))  # Guardar la imagen transformada en memoria

    def __getitem__(self, index):
        return self.preloaded_data[index]  # Retornar la imagen preprocesada directamente

class PreloadedImageFolder(torch.utils.data.Dataset):
    def __init__(self, root, label, transform=None):
        self.root = root
        self.transform = transform
        self.preloaded_data = []
        self.label = label  # Guardar la etiqueta de la clase
        
        # Obtener solo archivos de imagen en la carpeta
        self.samples = [os.path.join(root, f) for f in os.listdir(root) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
        
        for path in self.samples:
            sample = Image.open(path).convert("RGB")  # Cargar imagen
            
            if self.transform:
                sample = self.transform(sample)  # Aplicar la transformación
            
            self.preloaded_data.append((sample, self.label))  # Guardar imagen
            

    def __len__(self):
        return len(self.preloaded_data)

    def __getitem__(self, index):
        return self.preloaded_data[index]



Create or load the transformed dataset

In [ ]:
def load_dataset(dataroot, dataset_path, image_size):
    if os.path.exists(dataset_path):
        dataset_dir = torch.load(dataset_path)
    else:
        name = ""
        if IMAGENETTE in dataset_path:
            name = IMAGENETTE
        elif IMAGEWOOF in dataset_path:
            name = IMAGEWOOF

        print(f"Creating dataset {image_size}x{image_size} for {name}")
        
        classes = [dir for dir in os.listdir(dataroot) if os.path.isdir(os.path.join(dataroot, dir))]
        dataset_dir = {}
        print(classes)

        for label, dir in enumerate(classes):
            class_path = os.path.join(dataroot, dir)
            print("Creating dataset of", class_path)
            dataset_dir[dir] = PreloadedImageFolder(root=class_path,
                                    label=label,
                                    transform=transforms.Compose([
                                    transforms.Resize(image_size),
                                    transforms.CenterCrop(image_size),
                                    transforms.ToTensor(),
                                    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                                ]))

        # show_images_grid(dataset, dataloaderIW_64,"Training Images Imagewoof")
        print("Saving at: ", dataset_path)
        torch.save(dataset_dir, dataset_path)
    return dataset_dir

In [ ]:
path_transformed_dataset = os.path.join(GENERATED, TRANSFORMED_DATASET)
path_datarootImagewoofTrain = os.path.join(datarootImagewoof, "train")
path_datarootImagenetteTrain = os.path.join(datarootImagenette, "train")



In [ ]:
def load_all_dataloaders(resolutions, batch_size, path_transformed_dataset, path_dataroot_imagewoof, path_dataroot_imagenette, num_workers=0):
    """
    Carga dataloaders por clase para Imagewoof2 e Imagenette2 en distintas resoluciones.

    Usa las variables globales IMAGEWOOF e IMAGENETTE para nombrar los datasets.

    Args:
        resolutions (list): Lista de resoluciones (por ejemplo [64, 128, 256]).
        batch_size (int): Tamaño del batch.
        path_transformed_dataset (str): Carpeta donde se guardan los datasets transformados.
        path_dataroot_imagewoof (str): Ruta al dataset original de Imagewoof2.
        path_dataroot_imagenette (str): Ruta al dataset original de Imagenette2.
        num_workers (int): Número de workers para los dataloaders.

    Returns:
        dict: Diccionario con dataloaders por dataset y resolución.
              Ej: {("imagewoof2", 64): {...}, ("imagenette2", 128): {...}, ...}
    """
    dataloaders_path = os.path.join(path_transformed_dataset, "dataloaders.pt")
    if os.path.exists(dataloaders_path):
        dataloader_dicts = torch.load(dataloaders_path)
    else:
        dataloader_dicts = {}
        start_time = time.time()

        for image_size in resolutions:
            # --- Imagewoof ---
            path_dataset_imagewoof = os.path.join(path_transformed_dataset, f"{IMAGEWOOF}_{image_size}.pt")
            dataset_dir = load_dataset(path_dataroot_imagewoof, path_dataset_imagewoof, image_size)

            dataloader_key = (IMAGEWOOF, image_size)
            dataloader_dicts[dataloader_key] = {
                dir: torch.utils.data.DataLoader(dataset_dir[dir], batch_size=batch_size, shuffle=True, num_workers=num_workers)
                for dir in dataset_dir.keys()
            }

            # --- Imagenette ---
            path_dataset_imagenette = os.path.join(path_transformed_dataset, f"{IMAGENETTE}_{image_size}.pt")
            dataset_dir = load_dataset(path_dataroot_imagenette, path_dataset_imagenette, image_size)

            dataloader_key = (IMAGENETTE, image_size)
            dataloader_dicts[dataloader_key] = {
                dir: torch.utils.data.DataLoader(dataset_dir[dir], batch_size=batch_size, shuffle=True, num_workers=num_workers)
                for dir in dataset_dir.keys()
            }
        
        print("Saving at: ", path_transformed_dataset)
        torch.save(dataloader_dicts, dataloaders_path)
        print(f"\nTotal time creation of all the datasets: {time.time() - start_time:.2f} seconds")

    return dataloader_dicts


In [ ]:
resolutions = [64, 128, 256]

dataloader_dicts = load_all_dataloaders(
    resolutions=resolutions,
    batch_size=batch_size,
    path_transformed_dataset=path_transformed_dataset,
    path_dataroot_imagewoof=path_datarootImagewoofTrain,
    path_dataroot_imagenette=path_datarootImagenetteTrain
)

### 64x64 Images

In [ ]:
# Imagewoof2
#dataloaderIW_64  = dataloader_dicts[(IMAGEWOOF, 64)]

# Imagenette2
#dataloaderIN_64  = dataloader_dicts[(IMAGENETTE, 64)]


Initialize of the weights of the nodes

In [ ]:
# custom weights initialization called on ``netG`` and ``netD``
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## Models

In [ ]:
def create_model(model_class, ngpu, device, nz=None, verbose=False):
    """
    Crea e inicializa un modelo (generator o discriminator).

    Args:
        model_class (nn.Module): Clase del modelo a instanciar.
        ngpu (int): Número de GPUs disponibles.
        device (torch.device): Dispositivo ('cuda' o 'cpu').
        nz (int, optional): Tamaño del vector latente z, solo requerido para generadores.
        verbose (bool): Si es True, imprime el modelo.

    Returns:
        nn.Module: Instancia del modelo inicializado.
    """
    # Instanciar el modelo con o sin nz según sea necesario
    if nz is not None:
        model = model_class(ngpu, nz).to(device)
    else:
        model = model_class(ngpu).to(device)

    # Multi-GPU support
    if (device.type == 'cuda') and (ngpu > 1):
        model = nn.DataParallel(model, list(range(ngpu)))

    # Inicialización de pesos
    model.apply(weights_init)

    # Mostrar arquitectura si se desea
    if verbose:
        print(model)

    return model


### Generator class

#### Generator 64x64

In [ ]:
# Generator Code

class Generator_64(nn.Module):
    def __init__(self, ngpu, nz):
        super(Generator_64, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d( nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``

            nn.ConvTranspose2d( ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``

            nn.ConvTranspose2d( ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``
            
            nn.ConvTranspose2d( ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(nc) x 64 x 64``
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
netG_64 = create_model(Generator_64, ngpu, device, nz=nz, verbose=True)

#### Generator 128x128

In [ ]:

# Generator Code

class Generator_128(nn.Module):
    def __init__(self, ngpu, nz):
        super(Generator_128, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d( nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``
            
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``

            nn.ConvTranspose2d( ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 32 x 32``

            ####### NUEVA CAPA
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. ``(ngf) x 64 x 64``
            ####### FIN NUEVA CAPA

            nn.ConvTranspose2d(ngf // 2, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(nc) x 128 x 128``
        )

    def forward(self, input):
        return self.main(input)
    
                


In [ ]:
netG_128 = create_model(Generator_128, ngpu, device, nz=nz, verbose=True)


#### Generator 256x256

In [ ]:
# Generator Code

class Generator_256(nn.Module):
    def __init__(self, ngpu, nz):
        super(Generator_256, self).__init__()
        self.ngpu = ngpu
        self.nz = nz
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d( nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. ``(ngf*8) x 4 x 4``

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. ``(ngf*4) x 8 x 8``

            nn.ConvTranspose2d( ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. ``(ngf*2) x 16 x 16``

            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. ``(ngf) x 32 x 32``

            ####### NUEVA CAPA
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. ``(ngf // 2) x 64 x 64``

            nn.ConvTranspose2d(ngf // 2, nc, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 4),
            nn.ReLU(True),
            # state size. ``(ngf // 4) x 128 x 128``
            ####### FIN NUEVAS CAPAS


            nn.ConvTranspose2d(ngf // 4, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
            # state size. ``(nc) x 256 x 256``
        )

    def forward(self, input):
        return self.main(input)

#### Initialization

In [ ]:
netG_256 = create_model(Generator_256, ngpu, device, nz=nz, verbose=True)


### Discriminator class

#### Discriminator 64x64

In [ ]:
# Discriminator Code

class Discriminator_64(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator_64, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is ``(nc) x 64 x 64``
            # nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
netD_64  = create_model(Discriminator_64, ngpu, device, verbose=True)

#### Discriminator 128x128

In [ ]:
# Discriminator Code 
class Discriminator_128(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator_128, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            
            ####### NUEVA CAPA
            # input (nc) x 128 x 128 -> (ndf//2) x 64 x 64
            nn.Conv2d(nc, ndf // 2, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # input is ``(nc) x 64 x 64``
            ####### FIN NUEVA CAPA
            
            nn.Conv2d(ndf // 2, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 32 x 32``

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 16 x 16``

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 8 x 8``

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 4 x 4``
            
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, input):
        return self.main(input)

In [ ]:
netD_128 = create_model(Discriminator_128, ngpu, device, verbose=True)

#### Discriminator 256x256

In [ ]:
# Discriminator Code

class Discriminator_256(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator_256, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # NUEVA CAPA: input (nc) x 256 x 256 -> (ndf//2) x 128 x 128
            nn.Conv2d(nc, ndf // 2, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # input is ``(ndf//2) x 128 x 128``
            nn.Conv2d(ndf // 2, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf) x 64 x 64``

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*2) x 32 x 32``

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*4) x 16 x 16``

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*8) x 8 x 8``

            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. ``(ndf*16) x 4 x 4``

            nn.Conv2d(ndf * 16, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
            # state size. ``1 x 1 x 1``
        )

    def forward(self, input):
        return self.main(input)


#### Initialization

In [ ]:
netD_256 = create_model(Discriminator_256, ngpu, device, verbose=True)

### Optimizer

In [ ]:
# Initialize the ``BCELoss`` function
criterion = nn.BCELoss()

# Establish convention for real and fake labels during training
real_label = 1.
fake_label = 0.


## Different Models to train

In [ ]:
generators_64 = []

for nz in NZ_VALUES:
    # Create the generator
    netG = Generator_64(ngpu, nz).to(device)

    # Handle multi-GPU if desired
    if (device.type == 'cuda') and (ngpu > 1):
        netG = nn.DataParallel(netG, list(range(ngpu)))

    # Apply the ``weights_init`` function to randomly initialize all weights
    #  to ``mean=0``, ``stdev=0.02``.
    netG.apply(weights_init)
    generators_64.append((netG, nz))


generators_128 = []

for nz in NZ_VALUES:
    # Create the generator
    netG = Generator_128(ngpu, nz).to(device)

    # Handle multi-GPU if desired
    if (device.type == 'cuda') and (ngpu > 1):
        netG = nn.DataParallel(netG, list(range(ngpu)))

    # Apply the ``weights_init`` function to randomly initialize all weights
    #  to ``mean=0``, ``stdev=0.02``.
    netG.apply(weights_init)
    generators_128.append((netG, nz))

generators_256 = []

for nz in NZ_VALUES:
    # Create the generator
    netG = Generator_256(ngpu, nz).to(device)

    # Handle multi-GPU if desired
    if (device.type == 'cuda') and (ngpu > 1):
        netG = nn.DataParallel(netG, list(range(ngpu)))

    # Apply the ``weights_init`` function to randomly initialize all weights
    #  to ``mean=0``, ``stdev=0.02``.
    netG.apply(weights_init)
    generators_256.append((netG, nz))

In [ ]:
def create_new_model(model_list, num, netD, netG, lr, dataloader, dataloaderName, num_epochs, className, image_size, nz, betas=None, step_size=None, gama=None):
    netD_copy = copy.deepcopy(netD)
    netG_copy = copy.deepcopy(netG)
    fixed_noise = torch.randn(NUM_IMAGES_FOR_STEP, nz, 1, 1, device=device)

    if betas is None:
        betas = (beta1, 0.999)

    optimizerD = optim.Adam(netD_copy.parameters(), lr=lr, betas=(beta1, 0.999))
    optimizerG = optim.Adam(netG_copy.parameters(), lr=lr, betas=(beta1, 0.999))

    if step_size is None and gama is None:
        schedulerD = None
        schedulerG = None
    else:
        schedulerD = torch.optim.lr_scheduler.StepLR(optimizerD, step_size=step_size, gamma=gama)
        schedulerG = torch.optim.lr_scheduler.StepLR(optimizerG, step_size=step_size, gamma=gama)
    
    if num_epochs > 0:
        model_list.append([num, netD_copy, optimizerD, schedulerD, netG_copy, optimizerG, schedulerG, dataloader, dataloaderName, num_epochs, className, lr, fixed_noise, image_size, nz])

def create_all_models(mmodels_dict, generators_dict, discriminators_dict, img_sizes, classificator_dicts, dataloader_dicts, learning_rates, num_epochs):
    
    for img_size in img_sizes:
        for dbName, classificator in classificator_dicts.items():            
                model_list = []
                dataloaderName = f"{dbName}_{img_size}"
                dataloader = dataloader_dicts[(dbName, img_size)]
            
                num = 1
                for netG, nz in generators_dict[img_size]:
                    for lr in learning_rates:
                        for  dir in classificator.values():
                            dataloader_class = dataloader[dir]
                            create_new_model(model_list, num, discriminators_dict[img_size], netG, lr, dataloader_class, dataloaderName, num_epochs, dir, img_size, nz, step_size=1, gama=0.99)
                        num += 1
                models_dict[(dbName, img_size)] = model_list

img_sizes = [64, 128, 256]
classificator_dicts = {
    IMAGEWOOF: classificatorImagewoof,
    IMAGENETTE: classificatorImagenette
}
discriminators_dict = {
    64: netD_64,
    128: netD_128,
    256: netD_256
}

generators_dict = {
    64: generators_64,
    128: generators_128,
    256: generators_256
}

models_dict = {}
create_all_models(models_dict, generators_dict, discriminators_dict, img_sizes, classificator_dicts, dataloader_dicts, LEARNING_RATES, NUM_EPOCHS)



## Train Models

In [ ]:
# Training Loop

def save_movel(model, optimizer, scheduler, epoch, losses, filename, img_list=None, num_images=None, trained_time=None):
    if scheduler is not None:
        scheduler = scheduler.state_dict()

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler,
        'epoch': epoch,
        'losses': losses,
        'img_list': img_list,
        'num_images': num_images,
        'trainedTime': trained_time,
    }, filename)

def train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, trained_epochs, num_epochs, 
          image_size, nz, fixed_noise, D_losses=None, G_losses=None, img_list=None, num_images=None, trained_time=None):

    print(f"Training model: {num}. - {modelD_filename} - {modelG_filename}") 

    # Lists to keep track of progress
    if img_list is None:
        # Crear un tensor vacío con forma (0, 3, image_size, image_size)
        img_list = torch.empty((0, 3, image_size, image_size))
    if num_images is None:
        num_images = 0
    if G_losses is None:
        G_losses = []
    if D_losses is None:
        D_losses = []

    if trained_time is None:
        trained_time = 0
    start_time = time.time()

    iters = 0

    # For each epoch
    for epoch in range(trained_epochs + 1, num_epochs + 1):
        # For each batch in the dataloader
        for i, data in enumerate(dataloader, 0):
            ############################
            # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
            ###########################
            ## Train with all-real batch
            netD.zero_grad()
            # Format batch
            real_cpu = data[0].to(device)
            b_size = real_cpu.size(0)
            label = torch.full((b_size,), real_label, dtype=torch.float, device=device)
            # Forward pass real batch through D
            output = netD(real_cpu).view(-1)
            # Calculate loss on all-real batch
            errD_real = criterion(output, label)
            # Calculate gradients for D in backward pass
            errD_real.backward()
            D_x = output.mean().item()

            ## Train with all-fake batch
            # Generate batch of latent vectors
            noise = torch.randn(b_size, nz, 1, 1, device=device)
            # Generate fake image batch with G
            fake = netG(noise)
            label.fill_(fake_label)
            # Classify all fake batch with D
            output = netD(fake.detach()).view(-1)  # detach para no propagar gradientes a G
            # Calculate D's loss on the all-fake batch
            errD_fake = criterion(output, label)
            # Calculate the gradients for this batch, accumulated (summed) with previous gradients
            errD_fake.backward()
            D_G_z1 = output.mean().item()
            # Compute error of D as sum over the fake and the real batches
            errD = errD_real + errD_fake
            # Update D
            optimizerD.step()

            ############################
            # (2) Update G network: maximize log(D(G(z)))
            ###########################
            netG.zero_grad()
            # Generate new noise for G
            noise = torch.randn(b_size, nz, 1, 1, device=device)
            fake = netG(noise)
            label.fill_(real_label)  # fake labels are real for generator cost
            # Since we just updated D, perform another forward pass of all-fake batch through D
            output = netD(fake).view(-1)
            # Calculate G's loss based on this output
            errG = criterion(output, label)
            # Calculate gradients for G
            errG.backward()
            D_G_z2 = output.mean().item()
            # Update G
            optimizerG.step()

            # Save Losses for plotting later
            G_losses.append(errG.item())
            D_losses.append(errD.item())

            iters += 1

        if schedulerD is not None:
            schedulerD.step()  # Update the learning rate based on the scheduler

        # Save the model state
        if epoch % ROUNDS_PER_SAVE_IMAGES == 0 or epoch == num_epochs:
            trained_time += (time.time() - start_time)
            start_time = time.time()
            with torch.no_grad():
                fake = netG(fixed_noise).detach().cpu()
                # Normalizar imágenes de [-1,1] a [0,1]
                fake = (fake + 1) / 2
                img_list = torch.cat((fake, img_list), dim=0)  # Concat the new images
                num_images += 1

        if epoch % ROUNDS_PER_SAVE_MODELS == 0 or epoch == num_epochs:
            # Save the model state
            save_movel(netD, optimizerD, schedulerD, epoch, D_losses, modelD_filename)
            save_movel(netG, optimizerG, schedulerG, epoch, G_losses, modelG_filename, img_list, num_images, trained_time)
            # Output training stats
            print('[%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                % (epoch, num_epochs, errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

        if epoch == num_epochs:
            print(f"\tModel {num} trained for {num_epochs} epochs. Time spent: {trained_time:.2f} seconds")
            
        # Free unused memory to improve training consistency
        torch.cuda.empty_cache()
        gc.collect()




def load_all_models(models):

    trained_models = {}
    
    for model in models:

        num, netD, optimizerD, schedulerD, netG, optimizerG, schedulerG, dataloader, dataloaderName, num_epochs, className, lr, fixed_noise, image_size, nz = model

        lr_str = "%.5f" % lr
        
        modelD_filename = os.path.join(GENERATED, MODELS, dataloaderName, f"nz={nz}_lr={lr_str}_imgSize={image_size}_ModelD-{className}.pt")
        modelG_filename = os.path.join(GENERATED, MODELS, dataloaderName, f"nz={nz}_lr={lr_str}_imgSize={image_size}_ModelG-{className}.pt")

        if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):

            # Discriminator
            checkpoint = torch.load(modelD_filename)

            netD.load_state_dict(checkpoint['model_state_dict'])
            optimizerD.load_state_dict(checkpoint['optimizer_state_dict'])
            if checkpoint['scheduler_state_dict'] is not None:
                schedulerD.load_state_dict(checkpoint['scheduler_state_dict'])
            else:
                schedulerD = None
            trained_epochs = checkpoint['epoch']
            D_losses = checkpoint['losses']

            # Genetator
            checkpoint = torch.load(modelG_filename)

            netG.load_state_dict(checkpoint['model_state_dict'])
            optimizerG.load_state_dict(checkpoint['optimizer_state_dict'])
            if checkpoint['scheduler_state_dict'] is not None:
                schedulerG.load_state_dict(checkpoint['scheduler_state_dict'])
            else:
                schedulerG = None
            
            img_list = checkpoint['img_list']
            G_losses = checkpoint['losses']
            trained_time = checkpoint['trainedTime']
            num_images = checkpoint['num_images']
            epochs = checkpoint['epoch']


            print(f"Model {num}, loaded from saved file. {modelD_filename} - {modelG_filename}")
            

            if trained_epochs < num_epochs:
                print(f"\tModel trained for {trained_epochs}/{num_epochs}. Time spent: {trained_time} seconds")
                train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, 
                      trained_epochs, num_epochs, image_size, nz, fixed_noise, D_losses, G_losses, img_list, num_images, trained_time)
            else:
                print(f"\tModel trained for {trained_epochs}/{epochs}. Time spent: {trained_time} seconds")
            
        else:
            train(num, modelD_filename, netD, optimizerD, schedulerD, modelG_filename, netG, optimizerG, schedulerG, dataloader, 
                  0, num_epochs, image_size, nz, fixed_noise)
        
        if num not in trained_models:
            trained_models[num] = []

        trained_models[num].append([modelD_filename, modelG_filename, dataloaderName])
        
        torch.cuda.empty_cache()
        gc.collect()
                
    return trained_models



#### Train Imagewoof models 64x64

In [ ]:
trained_modelsIW_64 = load_all_models(models_dict[(IMAGEWOOF, 64)])

#### Train Imagenette models 64x64

In [ ]:
trained_modelsIN_64 = load_all_models(models_dict[(IMAGENETTE, 64)])

Train 

#### Train Imagewoof models 128x128

In [ ]:
trained_modelsIW_128 = load_all_models(models_dict[(IMAGEWOOF, 128)])

#### Train Imagenette models 128x128

In [ ]:
trained_modelsIN_128 = load_all_models(models_dict[(IMAGENETTE, 128)])

#### Train Imagewoof models 256x256

In [ ]:
trained_modelsIW_256 = load_all_models(models_dict[(IMAGEWOOF, 256)])

#### Train Imagenette models 256x256

In [ ]:
trained_modelsIN_128 = load_all_models(models_dict[(IMAGENETTE, 256)])


### Loss Graph 

Code of the function we will use to plot

In [ ]:
def plot_loss(models):
    for num, model in enumerate(models.keys(), 1):
        
        plt.figure(figsize=(10,5))
        
        for pair_model in models[model]: 
            modelD_filename, modelG_filename, datasetTitle = pair_model

            if IMAGENETTE in datasetTitle:
                classificator = classificatorImagenette_inv
                datasetName = IMAGENETTE
            elif IMAGEWOOF in datasetTitle:
                classificator = classificatorImagewoof_inv
                datasetName = IMAGEWOOF
            else:
                print("Invalid dataset")
                break

            
            if os.path.isfile(modelD_filename) and os.path.isfile(modelG_filename):
                checkpoint = torch.load(modelD_filename)
                D_losses = checkpoint['losses']

                checkpoint = torch.load(modelG_filename)
                G_losses = checkpoint['losses']

                patern = r"nz=(\d+)_lr=([\d.]+)_imgSize=(\d+)_ModelG-([a-z0-9]+)\.pt"
                coincidencia = re.match(patern, os.path.basename(modelG_filename))

                if coincidencia:
                    nz = int(coincidencia.group(1))
                    lr = float(coincidencia.group(2))
                    imgSize = int(coincidencia.group(3))
                    n = coincidencia.group(4)
                else:
                    print("La cadena no coincide con el patrón.")

                
                plt.plot(G_losses,label=f"G: {classificator[n]}")
                plt.plot(D_losses,label=f"D: {classificator[n]}")

        plt.title(f"{num}.{datasetName} Generator and Discriminator Loss During Training. nz={nz}, lr={lr}, imgSize={imgSize}")
        plt.xlabel("iterations")
        plt.ylabel("Loss")
        plt.legend()
        plt.show()

#### Imagiwoof 64x64:

In [ ]:
plot_loss(trained_modelsIW_64)

#### Imaginette 64x64

In [ ]:
plot_loss(trained_modelsIN_64)

#### Imagewoof 128x128

In [ ]:
plot_loss(trained_modelsIW_128)

#### Imagenette 128x128

In [ ]:
plot_loss(trained_modelsIN_128)

## Results

### Models Learning Progress

Code of the function we will use to visualize the results

In [ ]:
def show_image_results(models, img_size, increment=None):
    html_videos = []  # Para almacenar los videos en HTML

    for num, model in enumerate(models.keys(), 1):
        # Crear un tensor vacío con forma (0, 3, 64, 64)
        total_models = len(models[model])

        print("\n\nShowing Results of Model", num)
        for i in range(total_models):
            _, modelG_filename, datasetTitle = models[model][i]

            if os.path.isfile(modelG_filename):
                checkpoint = torch.load(modelG_filename)
                img_list = torch.empty((0, 3, img_size, img_size))
                model_images = checkpoint['img_list']
                num_images = checkpoint['num_images']
                # Only add images if num_images % increment == 0

                for i in range(1, num_images, increment):
                    # img_list = torch.cat((img_list, model_images[i].unsqueeze(0)), dim=0)
                    img_list = torch.cat((img_list, model_images[i*NUM_IMAGES_FOR_STEP: (i+1)*NUM_IMAGES_FOR_STEP]), dim=0)
                # img_list = torch.cat((img_list, model_images), dim=0)

                grid_img = vutils.make_grid(img_list, nrow=NUM_IMAGES_FOR_STEP, normalize=True)
                np_img = grid_img.permute(1, 2, 0).numpy()

                # Usa fig + ax para mantener control total sobre la figura
                fig, ax = plt.subplots(figsize=(20, 20))
                ax.imshow(np_img)
                ax.axis('off')
                if IMAGENETTE in datasetTitle:
                    classificator = classificatorImagenette_inv
                    datasetName = IMAGENETTE
                elif IMAGEWOOF in datasetTitle:
                    classificator = classificatorImagewoof_inv
                    datasetName = IMAGEWOOF
                else:
                    print("Invalid dataset")
                    break

                patern = r"nz=(\d+)_lr=([\d.]+)_imgSize=(\d+)_ModelG-([a-z0-9]+)\.pt"
                coincidencia = re.match(patern, os.path.basename(modelG_filename))

                if coincidencia:
                    nz = int(coincidencia.group(1))
                    lr = float(coincidencia.group(2))
                    imgSize = int(coincidencia.group(3))
                    n = coincidencia.group(4)
                else:
                    print("La cadena no coincide con el patrón.")

                # Añadir título a la figura
                ax.set_title(f"{num}.{datasetName} Generator evolution during training for {classificator[n]}. nz={nz}, lr={lr}, imgSize={imgSize}", fontsize=16)

                # Añadir texto por fila (una etiqueta por iteración)
                numbers = list(range(1, num_images, increment))[::-1]
                num_rows = len(list(range(1, num_images, increment)))
                img_height = img_size  # o 128 si estás generando imágenes más grandes

                for i in range(num_rows):
                    y = i * img_height + img_height // 2
                    ax.text(-10, y, f"Epoch {numbers[i]}", va='center', ha='right',
                            fontsize=12, color='white', backgroundcolor='black')

                plt.tight_layout()
                plt.show()



In [ ]:
increment = 50

#### Imagewoof 64x64

In [ ]:
show_image_results(trained_modelsIW_64, 64, increment)

#### Imagenette 64x64

In [ ]:
show_image_results(trained_modelsIN_64, 64, increment)

#### Imagewoof 128x128

In [ ]:
show_image_results(trained_modelsIW_128, 128, increment)

#### Imagenette 128x128

In [ ]:
show_image_results(trained_modelsIN_128, 128, increment)

### Comparison

Compare the real photos to the fake ones of each model

In [ ]:

def compare_results(models, dataloaderName, image_size):
    # Selección del dataloader
    if dataloaderName == IMAGENETTE:
        classificator = classificatorImagenette_inv
        datasetName = IMAGENETTE
        dataroot = datarootImagenette
    elif dataloaderName == IMAGEWOOF:
        classificator = classificatorImagewoof_inv
        datasetName = IMAGEWOOF
        dataroot = datarootImagewoof
    else:
        print("Not recognized dataloader")
        return

    transform = transforms.ToTensor()

    def load_real_images(folder_path, max_images=8):
        """Carga hasta max_images imágenes reales como tensores"""
        images = []
        count = 0
        for file in sorted(os.listdir(folder_path)):
            if file.lower().endswith(('.jpeg', '.jpg', '.png')):
                path = os.path.join(folder_path, file)
                try:
                    img = Image.open(path).convert('RGB')
                    img_tensor = transform(img)
                    images.append(img_tensor)
                    count += 1
                    if count == max_images:
                        break
                except Exception as e:
                    print(f"Error abriendo imagen: {path}. Error: {e}")
        return images

    for num, model_name in enumerate(models.keys(), 1):
        fig = plt.figure(figsize=(15, 15))

        generated_images = []
        real_images = []
        labels = []

        for _, modelG_filename, _ in models[model_name]:
            # Cargar checkpoint del modelo
            if not os.path.isfile(modelG_filename):
                continue

            checkpoint = torch.load(modelG_filename)
            model_images = checkpoint['img_list'][:8]
            generated_images.append(model_images)

            # Extraer info del nombre del modelo
            pattern = r"nz=(\d+)_lr=([\d.]+)_imgSize=(\d+)_ModelG-([a-z0-9]+)\.pt"
            match = re.match(pattern, os.path.basename(modelG_filename))
            if not match:
                print(f"Nombre de archivo no válido: {modelG_filename}")
                continue

            nz = int(match.group(1))
            lr = float(match.group(2))
            imgSize = int(match.group(3))
            class_code = match.group(4)

            labels.append(classificator.get(class_code, class_code))

            # Cargar imágenes reales
            real_folder = os.path.join(dataroot, "train", class_code)
            real_images += load_real_images(real_folder)

        # Concatenar imágenes generadas
        if len(generated_images) == 0 or len(real_images) == 0:
            print(f"Saltando {model_name} por falta de imágenes.")
            continue

        img_list = torch.cat(generated_images, dim=0)
        img_list_real = torch.stack(real_images)

        # Mostrar Figura 1: Imágenes generadas
        plt.subplot(1, 2, 1)
        plt.axis("off")
        plt.title(f"{num}.{datasetName} Generator Results\nnz={nz}, lr={lr}, imgSize={imgSize}")

        grid = vutils.make_grid(img_list, nrow=8, normalize=True)
        grid_np = grid.permute(1, 2, 0).cpu().numpy()
        plt.imshow(grid_np)

        # Añadir etiquetas por fila
        alto_total = grid_np.shape[0]
        alto_fila = alto_total / len(labels)
        for i, label in enumerate(labels):
            plt.text(-10, i * alto_fila + alto_fila / 2, label,
                     va='center', ha='right', fontsize=10,
                     bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

        # Mostrar Figura 2: Imágenes reales
        plt.subplot(1, 2, 2)
        plt.axis("off")
        plt.title(f"{num}. Real Images")

        grid_real = vutils.make_grid(img_list_real, nrow=8, padding=5, normalize=True)
        grid_real_np = np.transpose(grid_real.cpu().numpy(), (1, 2, 0))
        plt.imshow(grid_real_np)

        plt.tight_layout()
        plt.show()

#### Imagewoof Comparison 64x64

In [ ]:
compare_results(trained_modelsIW_64, IMAGEWOOF, 64)

#### Imagenette Comparison 64x64

In [ ]:
compare_results(trained_modelsIN_64, IMAGENETTE, 64)

#### Imagewoof Comparison 128x128

In [ ]:
compare_results(trained_modelsIW_128, IMAGEWOOF, 128)

#### Imagenette Comparison 128x128

In [ ]:
compare_results(trained_modelsIN_128, IMAGENETTE, 128)

# Classificators